# ObraSync Datos-Alfa

Este proyecto propone una solución de análisis y predicción del consumo de materiales, insumos y consumibles, basada en datos históricos, con el objetivo de optimizar la planificación de compras para obras, reduciendo faltantes y mejorando la eficiencia en el uso de recursos.


# Importacion de libreria

In [1]:
import pandas as pd

# Carga del Archivo para su analisis

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/Tec-IA-Proyectos-26/ObraSync_Datos_Alfa/refs/heads/main/DATOS/DATA_BASE.csv")
df.head()

,Descripcion,Marca,Compra Total,Ingreso Total,Pendiente,Estado,Fecha,Año
0,ESPATULA PINTOR LAMINADA CABO PLASTICO - 50 MM,BIASSONI,36,NaN,36.0,INCOMPLETO,F. 29-1-2026,2026
1,AEROSOL SMART PAINT AA BLANCO BRILLANTE 350ML...,DOBLE A - AE,72,72.0,0.0,FULL,F. 29-1-2026,2026
2,PANTALON CARGO DEL NORTE MUJER - PAMPERO VERDE...,PAMPERO,2,NaN,2.0,INCOMPLETO,F. 29-1-2026,2026
3,REGULADOR GLP C/1 MAN. (PROP. Y OTROS),FERROLAN,15,15.0,0.0,FULL,F. 29-1-2026,2026
4,AEROSOL USO GENERAL NEGRO MATE 250 GR,TEKBOND,60,NaN,60.0,INCOMPLETO,F. 29-1-2026,2026


Tratamiento de columnas

In [3]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 635 entries, 0 to 634
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   descripcion    629 non-null    str    
 1   marca          596 non-null    str    
 2   compra_total   628 non-null    str    
 3   ingreso_total  533 non-null    float64
 4   pendiente      597 non-null    float64
 5   estado         543 non-null    str    
 6   fecha          635 non-null    str    
 7   año            635 non-null    int64  
dtypes: float64(2), int64(1), str(5)
memory usage: 39.8 KB


Conteo de valores nulos por columna

In [4]:
df.isnull().sum()

descripcion        6
marca             39
compra_total       7
ingreso_total    102
pendiente         38
estado            92
fecha              0
año                0
dtype: int64

Imputacion valores nulos a "GENERICO" en columna "marca" para no perder los demas datos

In [5]:
df["marca"] = df["marca"].fillna("GENERICO")

Eliminacíon de filas con descripcion faltante ya que desconocemos que producto es

In [6]:
df.dropna(subset=["descripcion"], inplace=True)

En este bloque convertimos a numero estas columnas, y si no se puede convertir, se pone NaN (que luego se rellenara).

Se hace esto para asegurarnos que las columnas son numericas, y poder hacer calculos con ellas.

Tambien se rellena con 1 en compra_total e ingreso_total si faltan. La logica es que si existe descripción pero faltan cantidades, se asume que hubo una transacción completa de 1 unidad (comprada e ingresada), lo que resulta en pendiente 0

In [7]:
df['compra_total'] = pd.to_numeric(df['compra_total'], errors='coerce')
df['ingreso_total'] = pd.to_numeric(df['ingreso_total'], errors='coerce')
df['pendiente'] = pd.to_numeric(df['pendiente'], errors='coerce')

df['compra_total'] = df['compra_total'].fillna(1)
df['ingreso_total'] = df['ingreso_total'].fillna(1)

# Calcular pendiente si falta, usando compra_total e ingreso_total
df['pendiente'] = df['pendiente'].fillna(df['compra_total'] - df['ingreso_total'])

# Si por alguna razon sigue habiendo NaN, rellenar con 0 la columna pendiente, ya que si no se pudo calcular, se asume que no hay pendiente
df['pendiente'] = df['pendiente'].fillna(0)

Estandarizacion de fechas

In [8]:
df["fecha"].unique()

<StringArray>
[ 'F. 29-1-2026',  'F. 19-1-2026',  'F. 2-12-2025',  'F. 8-11-2025',
 'F. 18-10-2025',  'F. 3-10-2025', 'F. 28-07-2025', 'F. 28-06-2025',
  'F. 2-06-2025', 'F. 22-05-2025', 'F. 12-05-2025', 'F. 19-04-2025',
   'F. 23-01-25',   'F. 6-3-2026']
Length: 14, dtype: str

In [9]:
df["fecha"] = df["fecha"].str.replace('F. ', '', regex=False)
df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True, format='mixed')

In [10]:
# Verificar los valores nulos despues de las transformaciones
df.isnull().sum()

descripcion       0
marca             0
compra_total      0
ingreso_total     0
pendiente         0
estado           86
fecha             0
año               0
dtype: int64

Imputación en columna "estado" comparando valores de columnas numericas de la misma fila.
> se puede imputar el estado comparando compra_total, ingreso_total y pendiente


In [11]:
def imputacion_estado(row):

    compra = row["compra_total"]
    ingreso = row["ingreso_total"]
    pendiente = row["pendiente"]

    if compra == ingreso:
        return "FULL"
    elif compra == pendiente:
        return "INCOMPLETO"
    elif ingreso > 0 < pendiente:
        return "PARCIAL"
    else:
        return row["estado"]

df.loc[df["estado"].isna(), "estado"] = df[df["estado"].isna()].apply(imputacion_estado, axis=1)

In [12]:
df.isnull().sum()

descripcion      0
marca            0
compra_total     0
ingreso_total    0
pendiente        0
estado           0
fecha            0
año              0
dtype: int64

 Se verifica si hay filas duplicadas

In [13]:
print(f"Total de filas repetidas: {df.duplicated().sum()}")

Total de filas repetidas: 1


Se procede a eliminar las filas duplicadas

In [14]:
df.drop_duplicates()

,descripcion,marca,compra_total,ingreso_total,pendiente,estado,fecha,año
0,ESPATULA PINTOR LAMINADA CABO PLASTICO - 50 MM,BIASSONI,36.0,1.0,36.0,INCOMPLETO,2026-01-29,2026
1,AEROSOL SMART PAINT AA BLANCO BRILLANTE 350ML...,DOBLE A - AE,72.0,72.0,0.0,FULL,2026-01-29,2026
2,PANTALON CARGO DEL NORTE MUJER - PAMPERO VERDE...,PAMPERO,2.0,1.0,2.0,INCOMPLETO,2026-01-29,2026
3,REGULADOR GLP C/1 MAN. (PROP. Y OTROS),FERROLAN,15.0,15.0,0.0,FULL,2026-01-29,2026
4,AEROSOL USO GENERAL NEGRO MATE 250 GR,TEKBOND,60.0,1.0,60.0,INCOMPLETO,2026-01-29,2026
...,...,...,...,...,...,...,...,...
630,RUEDA MACIZA DE CARRETILLA 380 X 90,MECANOBRA,20.0,1.0,20.0,INCOMPLETO,2026-03-06,2026
631,ESCUADRA CON SOMBRERO BREMEN® 150X100MM DIN 875/1,BREMEN,10.0,10.0,0.0,FULL,2026-03-06,2026
632,PILA ENERGIZER MAX AAA BLX4 E92 MAX,ENERGIZER,100.0,14.0,86.0,PARCIAL,2026-03-06,2026
633,SELLADOR MS HIBRIDO BLANCO CARTUCHO 250ML/400GRS,TEKBOND,12.0,12.0,0.0,FULL,2026-03-06,2026


In [15]:
df.info()

<class 'pandas.DataFrame'>
Index: 629 entries, 0 to 634
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   descripcion    629 non-null    str           
 1   marca          629 non-null    str           
 2   compra_total   629 non-null    float64       
 3   ingreso_total  629 non-null    float64       
 4   pendiente      629 non-null    float64       
 5   estado         629 non-null    str           
 6   fecha          629 non-null    datetime64[us]
 7   año            629 non-null    int64         
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 44.2 KB


Observamos: Que existe una discontinuidad de indices.
Reindexamos el DataFrame para que los indices sean consecutivos, para evitar problemas en futuras operaciones que dependan de los indices, como los merges

In [16]:
df.reset_index(drop=True, inplace=True)

Limpieza de datos finalizada. Resultado:

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 629 entries, 0 to 628
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   descripcion    629 non-null    str           
 1   marca          629 non-null    str           
 2   compra_total   629 non-null    float64       
 3   ingreso_total  629 non-null    float64       
 4   pendiente      629 non-null    float64       
 5   estado         629 non-null    str           
 6   fecha          629 non-null    datetime64[us]
 7   año            629 non-null    int64         
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 39.4 KB


Dataset limpio y listo para exportar

In [18]:
df.to_csv("DATOS/DATA_BASE_CLEAN.csv", index=False)